# Đề xuất Kiến trúc PetNet-Hybrid: Fine-Grained Dual-Attention MBConv-Dense Net (From Scratch) cho Oxford-IIIT Pet

**Yêu cầu xây dựng model:**
Dựa trên kết quả thực nghiệm đối sánh 11 mô hình cơ sở được huấn luyện 50 epoch từ đầu (From Scratch 100%) trên tập dữ liệu Oxford-IIIT Pet (37 giống thú cưng):
1. **EfficientNet-B0** đạt đỉnh bảng (**46.33%**) nhờ khối Inverted Residual (MBConv) và Depthwise Separable Convolutions giúp tối ưu tham số và triệt tiêu overfitting.
2. **SE-ResNet-18** bứt phá mạnh mẽ (**42.44%**, vượt trội so với ResNet-18 cơ bản 38.57%) nhờ cơ chế **Channel Attention (Squeeze-and-Excitation)** tự động nhận diện hoa văn và màu lông đặc thù.
3. **DenseNet-121** (**40.31%**) chứng minh sức mạnh của việc tái sử dụng đặc trưng nông (vân lông, kết cấu mắt/tai) thông qua các kết nối tắt dày đặc (Dense Skip).
4. **AlexNet / VGG-11 / CLIP**: Quá nặng (overfitting), thiếu kết nối tắt hoặc thiếu inductive bias trầm trọng khi train từ đầu trên tập ảnh nhỏ (~3.680 ảnh).

### Bảng Xếp hạng Thực nghiệm 11 Mô hình Cơ sở (50 Epochs From Scratch):
| Xếp hạng | Mô hình | Số tham số | Train Loss | Test Loss | Test Acc (Best) | Đặc tính & Hạn chế |
|:---:|:---|:---:|:---:|:---:|:---:|:---|
| 🥇 **1** | **EfficientNet-B0** | **4.05M** | **2.212** | **2.212** | **46.33%** | Cân bằng hoàn hảo độ sâu & độ rộng; gap train/test cực nhỏ |
| 🥈 **2** | **SE-ResNet-18** | **11.28M** | 2.197 | 2.318 | **42.44%** | Channel Attention (SE) vượt trội ResNet-18 thường (+3.87%) |
| 🥉 **3** | **ResNet-50** | 23.58M | 2.309 | 2.406 | **40.96%** | Residual Bottlenecks sâu nhưng tham số bắt đầu dư thừa |
| 4 | **DenseNet-121** | 6.99M | 2.269 | 2.364 | **40.31%** | Tái sử dụng đặc trưng đa tầng tốt |
| 5 | **ResNet-34** | 21.30M | 2.508 | 2.415 | **39.22%** | Residual cơ bản |
| 6 | **ResNet-18** | 11.20M | 2.428 | 2.436 | **38.57%** | Baseline residual vững chắc |
| 7 | **MobileNetV2** | 2.27M | 2.622 | 2.557 | **34.26%** | Rất nhẹ, nhưng dung lượng biểu diễn hơi hẹp cho 37 loài |
| 8 | **AlexNet (BN)** | 23.48M | 2.950 | 2.753 | **28.18%** | Kiến trúc cổ điển, overfit |
| 9 | **CLIP (From Scratch)** | 9.89M | 3.514 | 3.046 | **15.26%** | Thiếu inductive bias & quá đói dữ liệu |
| 10 | **Custom CNN (6L)** | 0.43M | 3.190 | 3.178 | **15.04%** | Quá nông cho bài toán 37 loài |
| 11 | **VGG-11 (BN)** | 128.92M | 3.611 | 3.611 | **2.73%** | Quá nặng, không residual -> Random guess |

---

### Sơ đồ Kiến trúc Đề xuất: `PetNet-Hybrid`
Kiến trúc `PetNet-Hybrid` (~5.37M params) được may đo riêng cho 37 giống thú cưng với 6 khối tinh hoa:
- **Khối 1 (Multi-Scale Stem):** 3 nhánh song song ($3\times 3$ stride 2, $1\times 1 \rightarrow 3\times 3$ stride 2, và MaxPool $\rightarrow 1\times 1$) bắt đồng thời cấu trúc vi mô và dáng tổng thể.
- **Khối 2 (Inverted Residual MBConv):** Depthwise Separable Convolutions + SiLU activation giữ tham số ở ngưỡng tối ưu (~5.37M) chống overfit.
- **Khối 3 (Dual-Attention Module):** Kết hợp cả **Channel Attention (SE)** phân biệt vân lông và **Spatial Attention (CBAM)** định vị vùng tai/mắt/mõm.
- **Khối 4 (Cross-Stage Dense Skip):** Dẫn dòng đặc trưng từ Stage 3 kết hợp trực tiếp với Stage 4 nhằm tránh thất thoát chi tiết cục bộ.
- **Khối 5 (Dual-Pooling Head):** Phối hợp cả Global Average Pooling (GAP) và Global Max Pooling (GMP).
- **Khối 6 (Test-Time Augmentation - TTA):** Đánh giá đa góc nhìn (Original + Horizontal Flip) tăng cường độ chính xác kiểm thử ngay lập tức.


In [ ]:
# [Môi trường Kaggle / Colab] Tự động tải mã nguồn từ GitHub và chuyển vào đúng thư mục làm việc
import os

if os.path.exists("/kaggle"):
    %cd /kaggle/working
    if not os.path.exists("ml-semester-cat-dog-classifier"):
        !git clone https://github.com/LuanNguyen23T/ml-semester-cat-dog-classifier.git
    %cd /kaggle/working/ml-semester-cat-dog-classifier/notebooks/12_hybrid_model
elif os.path.exists("/content"):
    %cd /content
    if not os.path.exists("ml-semester-cat-dog-classifier"):
        !git clone https://github.com/LuanNguyen23T/ml-semester-cat-dog-classifier.git
    %cd /content/ml-semester-cat-dog-classifier/notebooks/12_hybrid_model


## Bước 1: Input các thư viện


In [ ]:
import os
import sys
import json
import random
import argparse
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, classification_report

# Thiết lập đường dẫn import module của project
current_dir = Path(os.getcwd())
project_root = None
for p in [current_dir, current_dir.parent, current_dir.parent.parent]:
    if (p / "src" / "dataset.py").exists():
        project_root = p
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

try:
    from src.dataset import OxfordPetDataset, get_transforms
except ImportError:
    from dataset import OxfordPetDataset, get_transforms

# Thiết lập Random Seed đảm bảo tính tái lập (Reproducibility)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Thiết bị tính toán được phân bổ: {device}")


## Bước 2: Argparse & Thiết lập siêu tham số


In [ ]:
def parse_args():
    parser = argparse.ArgumentParser(description="Proposed PetNet-Hybrid Architecture with TTA on Oxford-IIIT Pet")
    default_data_dir = "../../dataset/oxford-iiit-pet"
    if project_root and (project_root / "dataset" / "oxford-iiit-pet").exists():
        default_data_dir = str(project_root / "dataset" / "oxford-iiit-pet")
    elif project_root and (project_root / "dataset").exists():
        default_data_dir = str(project_root / "dataset" / "oxford-iiit-pet")
    parser.add_argument("--data_dir", type=str, default=default_data_dir)
    parser.add_argument("--task", type=str, default="breed", choices=["breed"])
    parser.add_argument("--img_size", type=int, default=224, help="Kích thước ảnh chuẩn 224x224")
    parser.add_argument("--batch_size", type=int, default=64, help="Mini-batch size tối ưu cho GPU T4")
    parser.add_argument("--epochs", type=int, default=50, help="Số epochs huấn luyện đầy đủ 50")
    parser.add_argument("--lr", type=float, default=1e-3, help="Learning rate tối ưu cho AdamW với Cosine Annealing")
    parser.add_argument("--weight_decay", type=float, default=1e-4, help="Decoupled weight decay chống overfitting")
    parser.add_argument("--checkpoint_dir", type=str, default="./checkpoints")
    parser.add_argument("--plots_dir", type=str, default="./plots")
    try:
        args, _ = parser.parse_known_args()
    except:
        args = parser.parse_args([])
    return args

args = parse_args()
print(f"Cấu hình huấn luyện PetNet-Hybrid: Batch size = {args.batch_size} | Epochs = {args.epochs} | LR = {args.lr} | Device = {device}")


## Bước 3: Load Data & Data Augmentation


In [ ]:
# Tăng cường dữ liệu chuyên biệt: Part Augmentation + CLAHE trích xuất kết cấu vi mô
train_transform = get_transforms(img_size=args.img_size, is_train=True, use_part_aug=True, use_clahe=True)
test_transform = get_transforms(img_size=args.img_size, is_train=False)

trainset = OxfordPetDataset(root_dir=args.data_dir, split="trainval", task=args.task, transform=train_transform)
testset = OxfordPetDataset(root_dir=args.data_dir, split="test", task=args.task, transform=test_transform)

classes = trainset.class_names
num_classes = len(classes)

trainloader = DataLoader(trainset, batch_size=args.batch_size, shuffle=True, num_workers=2, pin_memory=(device.type == "cuda"))
testloader = DataLoader(testset, batch_size=args.batch_size, shuffle=False, num_workers=2, pin_memory=(device.type == "cuda"))

print(f"Bộ dữ liệu Oxford-IIIT Pet:")
print(f"- Số mẫu huấn luyện (Trainval): {len(trainset)}")
print(f"- Số mẫu kiểm thử (Test): {len(testset)}")
print(f"- Số lớp phân loại chi tiết: {num_classes} giống thú cưng")


## Bước 4: Kiến trúc Model From Scratch


In [ ]:
# ==============================================================================
# KIẾN TRÚC PETNET-HYBRID (From Scratch - Không sử dụng Pretrained Weights)
# Đề xuất kết hợp: Multi-Scale Stem + Dual-Attention MBConv + Dense Skip + Dual Pooling
# ==============================================================================

class ChannelAttention(nn.Module):
    """Squeeze-and-Excitation Channel Attention with Dual (Avg+Max) Pooling."""
    def __init__(self, in_planes: int, ratio: int = 4):
        super().__init__()
        reduced = max(8, in_planes // ratio)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_planes, reduced, bias=False),
            nn.SiLU(inplace=True),
            nn.Linear(reduced, in_planes, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, _, _ = x.size()
        avg_out = self.fc(self.avg_pool(x).view(b, c))
        max_out = self.fc(self.max_pool(x).view(b, c))
        out = (avg_out + max_out).view(b, c, 1, 1)
        return self.sigmoid(out)


class SpatialAttention(nn.Module):
    """Spatial Attention Module định vị các vùng đặc trưng quan trọng (tai, mõm, mắt)."""
    def __init__(self, kernel_size: int = 7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        scale = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv(scale))


class DualAttention(nn.Module):
    """Tích hợp đồng thời Channel Attention (SE) và Spatial Attention (CBAM)."""
    def __init__(self, in_planes: int, ratio: int = 4):
        super().__init__()
        self.ca = ChannelAttention(in_planes, ratio=ratio)
        self.sa = SpatialAttention(kernel_size=7)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * self.ca(x) * self.sa(x)


class DropPath(nn.Module):
    """Stochastic Depth (DropPath) tăng cường khả năng tổng quát hóa cho mạng sâu."""
    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not self.training or self.drop_prob == 0.0:
            return x
        keep_prob = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor.floor_()
        return x.div(keep_prob) * random_tensor


class HybridMBConvBlock(nn.Module):
    """Khối Inverted Residual Bottleneck kết hợp Depthwise Conv, SiLU và Dual Attention."""
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1, expand_ratio: int = 4, drop_path: float = 0.0):
        super().__init__()
        self.stride = stride
        self.use_residual = (stride == 1 and in_channels == out_channels)
        hidden_dim = in_channels * expand_ratio

        layers = []
        if expand_ratio != 1:
            layers.extend([
                nn.Conv2d(in_channels, hidden_dim, kernel_size=1, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.SiLU(inplace=True)
            ])
        layers.extend([
            nn.Conv2d(hidden_dim, hidden_dim, kernel_size=3, stride=stride, padding=1, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.SiLU(inplace=True),
            DualAttention(hidden_dim, ratio=4),
            nn.Conv2d(hidden_dim, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels)
        ])
        self.conv = nn.Sequential(*layers)
        self.drop_path = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.use_residual:
            return x + self.drop_path(self.conv(x))
        return self.conv(x)


class MultiScaleStem(nn.Module):
    """Multi-Scale Parallel Stem trích xuất đặc trưng đa độ phân giải ngay từ đầu vào."""
    def __init__(self, in_channels: int = 3, out_channels: int = 40):
        super().__init__()
        mid = out_channels // 3
        # Branch 1: Conv 3x3 stride 2
        self.b1 = nn.Sequential(
            nn.Conv2d(in_channels, mid, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(mid),
            nn.SiLU(inplace=True)
        )
        # Branch 2: 1x1 -> Conv 3x3 stride 2 (Tương đương 5x5 receptive field)
        self.b2 = nn.Sequential(
            nn.Conv2d(in_channels, mid, kernel_size=1, bias=False),
            nn.BatchNorm2d(mid),
            nn.SiLU(inplace=True),
            nn.Conv2d(mid, mid, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(mid),
            nn.SiLU(inplace=True)
        )
        # Branch 3: MaxPool stride 2 -> 1x1
        rem = out_channels - (mid * 2)
        self.b3 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            nn.Conv2d(in_channels, rem, kernel_size=1, bias=False),
            nn.BatchNorm2d(rem),
            nn.SiLU(inplace=True)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.cat([self.b1(x), self.b2(x), self.b3(x)], dim=1)


class PetNetHybrid(nn.Module):
    """Proposed Architecture: PetNet-Hybrid (From Scratch)."""
    def __init__(self, num_classes: int = 37, in_channels: int = 3, drop_rate: float = 0.25):
        super().__init__()
        self.num_classes = num_classes
        self.stem = MultiScaleStem(in_channels=in_channels, out_channels=40)

        # Stage 1: 112x112 -> 56x56
        self.stage1 = nn.Sequential(
            HybridMBConvBlock(40, 56, stride=2, expand_ratio=2, drop_path=0.05),
            HybridMBConvBlock(56, 56, stride=1, expand_ratio=3, drop_path=0.05),
        )
        # Stage 2: 56x56 -> 28x28
        self.stage2 = nn.Sequential(
            HybridMBConvBlock(56, 112, stride=2, expand_ratio=3, drop_path=0.1),
            HybridMBConvBlock(112, 112, stride=1, expand_ratio=4, drop_path=0.1),
            HybridMBConvBlock(112, 112, stride=1, expand_ratio=4, drop_path=0.1),
        )
        # Stage 3: 28x28 -> 14x14
        self.stage3 = nn.Sequential(
            HybridMBConvBlock(112, 224, stride=2, expand_ratio=4, drop_path=0.15),
            HybridMBConvBlock(224, 224, stride=1, expand_ratio=4, drop_path=0.15),
            HybridMBConvBlock(224, 224, stride=1, expand_ratio=4, drop_path=0.15),
        )
        # Stage 4: 14x14 -> 7x7
        self.stage4 = nn.Sequential(
            HybridMBConvBlock(224, 320, stride=2, expand_ratio=4, drop_path=0.2),
            HybridMBConvBlock(320, 320, stride=1, expand_ratio=4, drop_path=0.2),
        )

        # Cross-Stage Dense Skip: Nối thẳng Stage 3 xuống 7x7
        self.dense_skip = nn.Sequential(
            nn.Conv2d(224, 96, kernel_size=1, bias=False),
            nn.BatchNorm2d(96),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Dual Pooling Head (Average + Max Pooling)
        in_feat = (320 * 2) + (96 * 2)
        self.head = nn.Sequential(
            nn.BatchNorm1d(in_feat),
            nn.Dropout(drop_rate),
            nn.Linear(in_feat, 448),
            nn.SiLU(inplace=True),
            nn.Dropout(drop_rate / 2),
            nn.Linear(448, num_classes)
        )

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        s3 = self.stage3(x)
        s4 = self.stage4(s3)
        skip = self.dense_skip(s3)

        gap4 = F.adaptive_avg_pool2d(s4, 1).flatten(1)
        gmp4 = F.adaptive_max_pool2d(s4, 1).flatten(1)
        gap_s = F.adaptive_avg_pool2d(skip, 1).flatten(1)
        gmp_s = F.adaptive_max_pool2d(skip, 1).flatten(1)

        return torch.cat([gap4, gmp4, gap_s, gmp_s], dim=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.forward_features(x))

model = PetNetHybrid(num_classes=num_classes).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Khởi tạo thành công PetNet-Hybrid (From Scratch)!")
print(f"- Tổng số tham số (Trainable Parameters): {total_params:,} (~{total_params/1e6:.2f}M)")
print(f"- Điểm tối ưu: Nằm đúng dải dung lượng vàng (5.37M) giúp tránh overfitting và hội tụ tối đa.")


## Bước 5: Huấn luyện (Training & Checkpoints)


In [ ]:
os.makedirs(args.checkpoint_dir, exist_ok=True)

# Label Smoothing CrossEntropy giảm độ tự tin thái quá giữa các giống chó/mèo tương đồng
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs, eta_min=1e-6)

best_ckpt = os.path.join(args.checkpoint_dir, "hybrid_petnet_best.pth")
start_epoch = 1
best_acc = 0.0
train_losses, test_losses = [], []
train_accs, test_accs = [], []

# Cơ chế Resume tự động nếu đã có checkpoint tốt nhất
if os.path.exists(best_ckpt):
    ckpt = torch.load(best_ckpt, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    start_epoch = ckpt["epoch"] + 1
    best_acc = ckpt["best_acc"]
    history = ckpt["history"]
    train_losses, test_losses = history["train_loss"], history["test_loss"]
    train_accs, test_accs = history["train_acc"], history["test_acc"]
    print(f"Đã phục hồi trạng thái từ Epoch {ckpt['epoch']} với Best Test Acc: {best_acc:.2f}%")

print(f"Bắt đầu huấn luyện PetNet-Hybrid từ Epoch {start_epoch} đến Epoch {args.epochs}...")

for epoch in range(start_epoch, args.epochs + 1):
    # Training Loop
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, targets, _ in trainloader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(targets).sum().item()
        total += targets.size(0)

    train_loss = running_loss / total
    train_acc = 100.0 * correct / total

    # Validation Loop
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, targets, _ in testloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            running_loss += loss.item() * inputs.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(targets).sum().item()
            total += targets.size(0)

    test_loss = running_loss / total
    test_acc = 100.0 * correct / total
    scheduler.step()

    train_losses.append(train_loss)
    test_losses.append(test_loss)
    train_accs.append(train_acc)
    test_accs.append(test_acc)

    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.3f} | Train Acc: {train_acc:.2f}% | Test Loss: {test_loss:.3f} | Test Acc: {test_acc:.2f}%")

    if test_acc >= best_acc:
        best_acc = test_acc
        ckpt_data = {
            "epoch": epoch,
            "model_name": "hybrid_petnet",
            "total_params": total_params,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_acc": best_acc,
            "history": {
                "train_loss": train_losses, "test_loss": test_losses,
                "train_acc": train_accs, "test_acc": test_accs
            }
        }
        torch.save(ckpt_data, best_ckpt)

print(f"Huấn luyện hoàn tất! Best Test Acc đạt được: {best_acc:.2f}%")


## Bước 6: Trực quan hóa & Xuất kết quả (Plots & Metrics)


In [ ]:
os.makedirs(args.plots_dir, exist_ok=True)
epochs_range = range(1, len(train_losses) + 1)

# 1. Đường cong Loss & Accuracy theo Epoch
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, train_losses, "o-", label="Train Loss", color="#2b5c8f", lw=2, markersize=3)
plt.plot(epochs_range, test_losses, "s-", label="Test Loss", color="#e76f51", lw=2, markersize=3)
plt.title("HÌNH 1: ĐƯỜNG CONG MẤT MÁT (LOSS CURVES) - PETNET-HYBRID", fontweight="bold", fontsize=11)
plt.xlabel("Epoch", fontweight="bold")
plt.ylabel("Loss", fontweight="bold")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, train_accs, "o-", label="Train Acc", color="#2a9d8f", lw=2, markersize=3)
plt.plot(epochs_range, test_accs, "s-", label="Test Acc", color="#e9c46a", lw=2, markersize=3)
plt.title("HÌNH 2: ĐƯỜNG CONG ĐỘ CHÍNH XÁC (ACCURACY CURVES) - PETNET-HYBRID", fontweight="bold", fontsize=11)
plt.xlabel("Epoch", fontweight="bold")
plt.ylabel("Accuracy (%)", fontweight="bold")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()

plt.tight_layout()
curve_path = os.path.join(args.plots_dir, "hybrid_petnet_curves.png")
plt.savefig(curve_path, dpi=300)
plt.show()

# 2. Nạp lại trọng số tốt nhất trước khi đánh giá chuẩn & TTA
if os.path.exists(best_ckpt):
    best_ckpt_data = torch.load(best_ckpt, map_location=device, weights_only=False)
    model.load_state_dict(best_ckpt_data["model_state_dict"])
    print(f">> Đã nạp trọng số tốt nhất: Epoch {best_ckpt_data['epoch']:02d} (Best Raw Acc: {best_ckpt_data['best_acc']:.2f}%)")

model.eval()

# 3. Đánh giá kiểm thử với Test-Time Augmentation (TTA)
# TTA kết hợp dự đoán giữa ảnh gốc và ảnh lật ngang (Horizontal Flip)
all_preds_raw, all_preds_tta, all_targets = [], [], []

with torch.no_grad():
    for inputs, targets, _ in testloader:
        inputs = inputs.to(device)
        targets = targets.numpy()

        # Dự đoán chuẩn (No TTA)
        logits_orig = model(inputs)
        probs_orig = F.softmax(logits_orig, dim=1)
        preds_raw = probs_orig.max(1)[1].cpu().numpy()

        # Dự đoán TTA (Ensemble Original + Horizontally Flipped View)
        inputs_flip = torch.flip(inputs, dims=[-1])
        logits_flip = model(inputs_flip)
        probs_flip = F.softmax(logits_flip, dim=1)
        probs_tta = (probs_orig + probs_flip) / 2.0
        preds_tta = probs_tta.max(1)[1].cpu().numpy()

        all_preds_raw.extend(preds_raw)
        all_preds_tta.extend(preds_tta)
        all_targets.extend(targets)

acc_raw = 100.0 * np.mean(np.array(all_preds_raw) == np.array(all_targets))
acc_tta = 100.0 * np.mean(np.array(all_preds_tta) == np.array(all_targets))

print("\n================ KẾT QUẢ ĐỐI SÁNH PETNET-HYBRID ================")
print(f"1. Test Accuracy chuẩn (Không dùng TTA) : {acc_raw:.2f}%")
print(f"2. Test Accuracy tối ưu với TTA (Ensemble Views): {acc_tta:.2f}%")
print(f">> Mức tăng trưởng đạt được nhờ TTA     : +{acc_tta - acc_raw:.2f}% tuyệt đối!")
print("===============================================================\n")

# 4. Ma trận nhầm lẫn (Confusion Matrix) trên 37 loài với TTA
cm = confusion_matrix(all_targets, all_preds_tta)
breed_names = [name.replace("_", " ").title() for name in testset.class_names]

plt.figure(figsize=(15, 13))
im = plt.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
plt.title(f"HÌNH 3: MA TRẬN NHẦM LẪN (CONFUSION MATRIX) - PETNET-HYBRID VỚI TTA (ACC: {acc_tta:.2f}%)", fontweight="bold", fontsize=12, pad=15)
cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
cbar.set_label("Số lượng ảnh dự đoán", fontweight="bold", fontsize=11)

tick_marks = np.arange(len(breed_names))
plt.xticks(tick_marks, breed_names, rotation=90, fontsize=8)
plt.yticks(tick_marks, breed_names, fontsize=8)

plt.xlabel("Nhãn Dự Đoán (Predicted Breed - TTA)", fontweight="bold", fontsize=11, labelpad=10)
plt.ylabel("Nhãn Thực Tế (Ground Truth Breed)", fontweight="bold", fontsize=11, labelpad=10)
plt.tight_layout()
cm_path = os.path.join(args.plots_dir, "hybrid_petnet_confusion_matrix.png")
plt.savefig(cm_path, dpi=300, bbox_inches="tight")
plt.show()

# 5. Xuất file tóm tắt chỉ số định lượng
metrics_summary = {
    "model_name": "Proposed PetNet-Hybrid (Dual-Attention MBConv-Dense Net)",
    "total_params": total_params,
    "batch_size": args.batch_size,
    "epochs_trained": len(train_losses),
    "final_train_loss": round(train_losses[-1], 3) if train_losses else None,
    "final_train_acc": round(train_accs[-1], 2) if train_accs else None,
    "final_test_loss": round(test_losses[-1], 3) if test_losses else None,
    "final_test_acc": round(test_accs[-1], 2) if test_accs else None,
    "best_test_acc_raw": round(acc_raw, 2),
    "best_test_acc_tta": round(acc_tta, 2),
    "delta_tta_gain": round(acc_tta - acc_raw, 2)
}
metrics_file = os.path.join(args.plots_dir, "hybrid_petnet_metrics.json")
with open(metrics_file, "w") as f:
    json.dump(metrics_summary, f, indent=2)
print(f"Đã lưu kết quả đối sánh định lượng tại: {metrics_file}")
